# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane (confirmed, not switching):** Lane 2 – Refresh / Content Opportunity Scoring.

**Same data as my Week 3 contract:** March 2026, one row per page, decision moment = end of 15 March. Signals only use 1–15 March, and the label (`is_declining`: daily impressions in 16–31 March fall below 80% of 1–15 March) only uses 16–31 March. The shared code lives in `work/scripts/lane_data.py`, so my Week 5 model gets exactly the same rows, label and split.

**My rule idea, before checking anything:** *"A page is worth a look if people see it on page one but don't click it as much as pages at that position usually get clicked, and it's still young enough to be settling."*

That idea leans on two signals, and both sit behind real FlyRank flags from the session:
1. **Staleness** (the refresh flags). I measure it as content age in days on 15 March, from `content_created_date`.
2. **CTR vs. position** (the CTR-fix logic). This is a page's CTR minus the median CTR of all pages in the same position bucket.

Before encoding anything, I check both with a bucket table.

In [1]:
import sys
sys.path.insert(0, "../scripts")
import json
import numpy as np
import pandas as pd
import lane_data as L

con = L.connect()
frame = L.build_frame(con)
print(f"Pages: {len(frame):,} | clients: {frame['client_hash_id'].nunique()} | declining base rate: {frame['is_declining'].mean():.3f}")

C:\Users\Itachi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages: 92,548 | clients: 40 | declining base rate: 0.327


### Signal check 1 – Staleness (content age) → decline rate

In [2]:
age_table = (frame.assign(age_bucket=pd.cut(frame["age_days"], [-1, 30, 90, 180, 365, 1000],
                                              labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]))
             .groupby("age_bucket", observed=True)["is_declining"]
             .agg(n="size", decline_rate="mean").round(3))
age_table

,n,decline_rate
age_bucket,,
0-30d,8268,0.227
31-90d,21919,0.367
91-180d,18190,0.428
181-365d,34593,0.289
365d+,9578,0.267


**Verdict: MIXED.** The refresh-flag idea says older = more likely to decline. That only holds up to about 6 months: decline climbs from 0–30 days to 91–180 days (the peak). After that it falls, and pages older than 180 days decline *less* than the base rate. So "stale" as in "old" is not the risk. The risky window is **31–180 days**, when a page is still settling into its rankings. My rule only uses that window.

### Signal check 2 – CTR vs. position → decline rate
Expected CTR = the median CTR of pages in the same position bucket. On page two and beyond, the median CTR is 0, so "below expected" can't happen there.

In [3]:
frame["ctr_vs_expected"] = np.select(
    [frame["ctr_h1"] == 0, frame["ctr_gap"] < 0],
    ["zero_ctr", "below_expected"], "at_or_above_expected")

print("Median CTR by position bucket:")
print(frame.groupby("pos_bucket", observed=True)["ctr_h1"].median().round(4).to_string())

ctr_table = (frame.groupby(["pos_bucket", "ctr_vs_expected"], observed=True)["is_declining"]
                  .agg(n="size", decline_rate="mean").round(3))
ctr_table

Median CTR by position bucket:
pos_bucket
1-3      0.0021
3-5      0.0022
5-10     0.0010
10-20    0.0000
20+      0.0000


n  decline_rate
pos_bucket ctr_vs_expected                          
1-3        at_or_above_expected   5557         0.211
           below_expected         1960         0.360
           zero_ctr               3596         0.395
3-5        at_or_above_expected   8188         0.225
           below_expected         3381         0.357
           zero_ctr               4805         0.391
5-10       at_or_above_expected  14086         0.280
           below_expected         1480         0.456
           zero_ctr              12604         0.412
10-20      at_or_above_expected   7237         0.268
           zero_ctr               9255         0.329
20+        at_or_above_expected   7239         0.424
           zero_ctr              13157         0.317

**Verdict: CONFIRMED (on page one only).** In positions 1–10, pages with zero CTR or below-expected CTR decline noticeably more (about 0.36–0.46) than pages at or above the expected CTR (about 0.21–0.28). The pattern holds in all three page-one buckets. Past position 10 it doesn't apply, and in position 20+ it even reverses. So the rule only uses it for pages on page one.

**My rule in plain words (after the checks):** a page scores **2 points** if it sits on page one (average position ≤ 10) but its CTR is below the median for its position. It scores **1 point** if it is 31–180 days old. Points are multiplied by log impressions, so bigger pages come first when points tie.

**Reason codes (one per page):** `ctr_below_position_young_page`, `ctr_below_position`, `young_page_settling`, `no_flag`.

**Action labels:** `rewrite_title_and_snippet` (a CTR problem on page one), `monitor` (young page only), `no_action`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
scored = L.baseline_score(frame)
queue = (scored.sort_values("baseline_score", ascending=False, kind="stable")
               .reset_index(drop=True))
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

import os
os.makedirs("../outputs", exist_ok=True)
cols = ["rank", "client_hash_id", "content_hash_id", "baseline_score", "reason_code", "action",
        "imp_h1", "ctr_h1", "avg_pos_h1", "age_days"]
queue[cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue):,} ranked pages to work/outputs/baseline_action_score.csv")
print(queue["reason_code"].value_counts().to_string())

Wrote 92,548 ranked pages to work/outputs/baseline_action_score.csv
reason_code
no_flag                          38387
young_page_settling              26335
ctr_below_position               14006
ctr_below_position_young_page    13820


### How good is the rule? (the number my Week 5 model must beat)
Evaluated on the **same client-grouped test split** that Week 5 will use (seed 42, 25% of clients held out). I also show all pages for context.

In [5]:
from sklearn.metrics import roc_auc_score
train, test = L.client_split(scored)

rows = []
for name, d in [("test clients (the comparison)", test), ("all pages", scored)]:
    rows.append({"slice": name, "n": len(d), "base_rate": d["is_declining"].mean(),
                 **{f"P@{k}": L.precision_at_k(d["baseline_score"], d["is_declining"], k) for k in (50, 100, 500)},
                 "roc_auc": roc_auc_score(d["is_declining"], d["baseline_score"])})
metrics = pd.DataFrame(rows).round(3)

with open("../outputs/w04_baseline_metrics.json", "w") as fh:
    json.dump({"rule": "2*(pos<=10 & ctr below position median) + 1*(age 31-180d), x log impressions",
               "split": "GroupShuffleSplit by client, test_size=0.25, seed=42",
               "results": metrics.to_dict(orient="records")}, fh, indent=2)
metrics

,slice,n,base_rate,P@50,P@100,P@500,roc_auc
0,test clients (the comparison),24886,0.405,0.38,0.44,0.438,0.589
1,all pages,92548,0.327,0.36,0.37,0.404,0.584


**Reading it honestly:** the rule separates decliners from non-decliners only a little better than chance overall (ROC AUC about 0.59). Its **top 50 is actually below the test base rate**. Sorting ties by impressions pushes the biggest pages to the top, and big pages don't decline more often than small ones. That's a weak baseline, but a truthful one, and it's frozen now: Week 5 has to beat these exact numbers on this exact split.

## 3. Top-10 review

For each of the top ten: the action, why it's there, and what would make it wrong. IDs are shortened pseudonyms. `is_declining` is shown **only for review** and was never an input to the rule.

In [6]:
top10 = queue.head(10).copy()
top10["page"] = "…" + top10["content_hash_id"].str[-6:]
top10[["rank", "page", "action", "reason_code", "imp_h1", "clk_h1", "ctr_h1", "avg_pos_h1",
       "ctr_gap", "age_days", "momentum_h1", "is_declining"]].round(4)

,rank,page,action,reason_code,imp_h1,clk_h1,ctr_h1,avg_pos_h1,ctr_gap,age_days,momentum_h1,is_declining
0,1,…64dea5,rewrite_title_and_snippet,ctr_below_position_young_page,91474.0,185.0,0.0020,4.2087,-0.0002,89,1.1301,0
1,2,…ae744a,rewrite_title_and_snippet,ctr_below_position_young_page,86860.0,51.0,0.0006,5.9538,-0.0004,89,0.9465,1
2,3,…996314,rewrite_title_and_snippet,ctr_below_position_young_page,83715.0,133.0,0.0016,3.5184,-0.0006,89,1.0601,0
3,4,…3fca87,rewrite_title_and_snippet,ctr_below_position_young_page,55680.0,15.0,0.0003,6.6820,-0.0007,67,2.4291,1
4,5,…ecc69a,rewrite_title_and_snippet,ctr_below_position_young_page,51513.0,96.0,0.0019,4.2468,-0.0003,62,0.8088,0
5,6,…0afc2d,rewrite_title_and_snippet,ctr_below_position_young_page,49619.0,8.0,0.0002,9.8177,-0.0008,177,1.2369,0
6,7,…8da489,rewrite_title_and_snippet,ctr_below_position_young_page,49180.0,79.0,0.0016,4.4534,-0.0006,53,0.5283,0
7,8,…ec072d,rewrite_title_and_snippet,ctr_below_position_young_page,46199.0,11.0,0.0002,5.3002,-0.0008,89,1.0434,1
8,9,…ff1dc2,rewrite_title_and_snippet,ctr_below_position_young_page,44610.0,61.0,0.0014,4.0579,-0.0008,94,0.5756,0
9,10,…1785be,rewrite_title_and_snippet,ctr_below_position_young_page,43871.0,69.0,0.0016,2.9293,-0.0005,89,1.0466,1


| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | rewrite_title_and_snippet | ~91k impressions at position 4.2 with 0.20% CTR, 89 days old | Its CTR is only 0.02 points under the bucket median, basically at expected. If the query mix is informational (answer shown in the SERP), a rewrite won't change clicks |
| 2 | rewrite_title_and_snippet | ~87k impressions at position 6.0, 51 clicks (0.06% CTR), 89 days old | If the impressions come from one broad head query where it's shown but irrelevant, the fix is the target query, not the title |
| 3 | rewrite_title_and_snippet | ~84k impressions at position 3.5, 0.16% CTR, 89 days old | Its week-2 impressions are steady (momentum 1.06). If it's just a stable page with a normal CTR for its query type, it's a false alarm |
| 4 | rewrite_title_and_snippet | ~56k impressions at position 6.7, only 15 clicks, 67 days old | Impressions more than doubled in week 2 (momentum 2.4). A spike like this usually falls back on its own, so "decline" may just be the spike ending, not something a title fix addresses |
| 5 | rewrite_title_and_snippet | ~52k impressions at position 4.2, 0.19% CTR, 62 days old | CTR gap is tiny (−0.03 pts). It's on the list mainly because it's big, not because it's clearly underperforming |
| 6 | rewrite_title_and_snippet | ~50k impressions at position 9.8, 8 clicks, 177 days old | Position 9.8 is on the edge of page one, and the expected CTR there is shaky. It's also about to leave my 31–180 day window |
| 7 | rewrite_title_and_snippet | ~49k impressions at position 4.5, 0.16% CTR, 53 days old | Week 2 already dropped to about half of week 1 (momentum 0.53). It may already be sliding for a ranking reason a title rewrite won't fix |
| 8 | rewrite_title_and_snippet | ~46k impressions at position 5.3, 11 clicks, 89 days old | If a SERP feature (images, AI answer) sits above it, low CTR is the SERP's fault, not the snippet's |
| 9 | rewrite_title_and_snippet | ~45k impressions at position 4.1, 0.14% CTR, 94 days old | Momentum 0.58 again: an ongoing drop that the rule sees as a CTR problem |
| 10 | rewrite_title_and_snippet | ~44k impressions at position 2.9, 0.16% CTR, 89 days old | At position 2.9, 0.16% CTR is very low, which points at a mismatch between query and page. If it ranks for an unrelated query, the right action is re-targeting, not a snippet tweak |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks I can see:**
- **The CTR gaps at the top are tiny** (−0.02 to −0.08 percentage points). All ten are "below expected" by a hair and are ranked high only because they're huge. The rule rewards size more than underperformance, which is why its top-50 precision falls below the base rate.
- **Seven of the top ten come from the same client, and five are exactly 89 days old.** That looks like one publishing batch from one big client dominating the list, not ten independent problems. A per-client cap on the queue would fix this.
- **Momentum is ignored.** #4 was spiking, and #7 and #9 were already falling. The rule treats them the same, but they need different actions.
- **Only 4 of the top 10 actually declined** (`is_declining` above), in line with the weak P@50.

**Leakage check (all pass):**

In [7]:
rule_inputs = ["avg_pos_h1", "ctr_gap", "ctr_h1", "age_days", "log_imp_h1"]
future_or_label = ["imp_h2", "is_declining", "trend_direction", "trend_pct",
                   "content_updated_date", "last_optimized_date"]

print("Rule inputs:", rule_inputs)
print("Any future/label column among inputs? ", any(c in rule_inputs for c in future_or_label))
print("imp_h2 (future window) still in frame?", "imp_h2" in frame.columns)

# Product-decision / future dates in dim_content: how many are AFTER the decision moment?
dates = con.sql(f"""
    SELECT ROUND(AVG((content_updated_date > DATE '{L.DECISION_DATE}')::INT), 3) AS updated_after_decision,
           ROUND(AVG((last_optimized_date  > DATE '{L.DECISION_DATE}')::INT), 3) AS optimized_after_decision,
           ROUND(AVG((content_created_date > DATE '{L.DECISION_DATE}')::INT), 3) AS created_after_decision
    FROM {L.DIM_CONTENT}
    WHERE content_hash_id IN (SELECT content_hash_id FROM frame)
""").df()
dates

Rule inputs: ['avg_pos_h1', 'ctr_gap', 'ctr_h1', 'age_days', 'log_imp_h1']
Any future/label column among inputs?  False
imp_h2 (future window) still in frame? False


,updated_after_decision,optimized_after_decision,created_after_decision
0,0.808,1.0,0.0


- The rule uses only 1–15 March signals and the creation date. None of its inputs is the label or a future-window column.
- `content_updated_date` and `last_optimized_date` are **deliberately not used**. Most of them are dated *after* 15 March, so they describe edits that hadn't happened yet on the decision day. Using them would be a future-information leak. `content_created_date` is always before the decision moment, so age is safe.
- One small note: the "expected CTR" medians are computed over all pages' 1–15 March data (train and test clients). This uses no label or future data, but a stricter version would compute them on training clients only.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.